XGboost

In [ ]:
# train_xgb_model.py
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import joblib

# ===== 데이터 로드 =====
df = pd.read_csv("../data/train_dataset.csv")
y = df["추정일간클릭수"]

# ===== 인코딩 =====
goodid_encoder = LabelEncoder().fit(df["good_id"])
df["good_id_enc"] = goodid_encoder.transform(df["good_id"])

# weekday는 이미 숫자라 그대로 사용
X = df[["discount_price", "최근4주클릭수_비율", "weekday", "good_id_enc"]]

# ===== 학습 =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# XGBoost 회귀 모델 정의
xgb_model = xgb.XGBRegressor(
    n_estimators=300,       # 트리 개수
    learning_rate=0.1,      # 학습률
    max_depth=6,            # 트리 깊이
    subsample=0.8,          # 샘플링 비율
    colsample_bytree=0.8,   # 피처 샘플링 비율
    random_state=42,
    n_jobs=-1
)

# 학습
xgb_model.fit(X_train, y_train)

# 예측 및 평가
y_pred = xgb_model.predict(X_test)
print("테스트셋 R²:", r2_score(y_test, y_pred))
print("테스트셋 MSE:", mean_squared_error(y_test, y_pred))

# ===== 모델과 컬럼 구조, 인코더 함께 저장 =====
xgb_package = {
    "model": xgb_model,
    "columns": list(X.columns),
    "goodid_encoder": goodid_encoder
}
joblib.dump(xgb_package, "../models/xgb_model.pkl")

테스트셋 R²: 0.8931889785219017
테스트셋 MSE: 15003.416059417128


['../models/xgb_model.pkl']

gam

In [11]:
# train_gam_model.py
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from pygam import LinearGAM, s, f
import joblib

df = pd.read_csv("../data/train_dataset.csv")
y = df["추정일간클릭수"]

# ===== 인코딩 =====
goodid_encoder = LabelEncoder().fit(df["good_id"])
df["good_id_enc"] = goodid_encoder.transform(df["good_id"])

# weekday는 이미 숫자라 그대로 사용
X = df[["discount_price", "최근4주클릭수_비율", "weekday", "good_id_enc"]]

# ===== 학습 =====
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gam = LinearGAM(
    s(0) + s(1) + f(2) + f(3)   # 가격, 최근4주클릭수는 spline / weekday, good_id는 factor
).fit(X_train, y_train)

y_pred = gam.predict(X_test)
print("테스트셋 R²:", r2_score(y_test, y_pred))
print("테스트셋 MSE:", mean_squared_error(y_test, y_pred))

# ===== 모델과 컬럼 구조, 인코더 함께 저장 =====
gam_package = {
    "model": gam,
    "columns": list(X.columns),
    "goodid_encoder": goodid_encoder
}
joblib.dump(gam_package, "../models/gam_model.pkl")

테스트셋 R²: 0.7912246999454718
테스트셋 MSE: 29326.025032819565


['../models/gam_model.pkl']

nlp

In [12]:
# train_mlp_model.py
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib

# 데이터 로드
df = pd.read_csv("../data/train_dataset.csv")
y = df["추정일간클릭수"].values

# ===== pum_id 인코딩 =====
pumid_encoder = LabelEncoder().fit(df["pum_id"])
df["pum_id_enc"] = pumid_encoder.transform(df["pum_id"])

# weekday는 이미 숫자라 그대로 사용
X = df[["1개당가격", "최근4주클릭수_비율", "weekday", "pum_id_enc"]]

# ===== 스케일링 =====
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 텐서 변환
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

class MLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super(MLPRegressor, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = MLPRegressor(input_dim=X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

# 학습 루프
epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# 평가
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor).numpy()

print("테스트셋 R²:", r2_score(y_test, y_pred))
print("테스트셋 MSE:", mean_squared_error(y_test, y_pred))

# ===== 모델과 스케일러, 인코더 저장 =====
package = {
    "model_state": model.state_dict(),
    "scaler": scaler,
    "input_dim": X_train.shape[1],
    "columns": list(X.columns),
    "pumid_encoder": pumid_encoder
}
joblib.dump(package, "../models/nn_model.pkl")

Epoch 10/200, Loss: 173382.8750
Epoch 20/200, Loss: 110793.1641
Epoch 30/200, Loss: 66486.3594
Epoch 40/200, Loss: 44183.7578
Epoch 50/200, Loss: 41861.0703
Epoch 60/200, Loss: 41422.8125
Epoch 70/200, Loss: 40785.5234
Epoch 80/200, Loss: 40297.1055
Epoch 90/200, Loss: 39883.2109
Epoch 100/200, Loss: 39546.6875
Epoch 110/200, Loss: 39267.8516
Epoch 120/200, Loss: 39069.2500
Epoch 130/200, Loss: 38922.0977
Epoch 140/200, Loss: 38779.9180
Epoch 150/200, Loss: 38657.2578
Epoch 160/200, Loss: 38559.2578
Epoch 170/200, Loss: 38469.8008
Epoch 180/200, Loss: 38381.2344
Epoch 190/200, Loss: 38297.6211
Epoch 200/200, Loss: 38216.0430
테스트셋 R²: 0.7725110561574677
테스트셋 MSE: 31954.673086679108


['../models/nn_model.pkl']